In [15]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100

In [20]:
# ============================================================
# 1) Initialization
# ============================================================
def initialize_gray_scott(N, dt, Du=0.16, Dv=0.08, f=0.035, k=0.060, dx=1.0, noise=0.0):
    """
    Gray-Scott model in 2D:

        u_t = Du * Lap(u) - u*v^2 + f*(1-u)
        v_t = Dv * Lap(v) + u*v^2 - (f+k)*v

    We simulate on a square grid with periodic boundary conditions
    in BOTH x- and y-directions (recommended for pattern formation).

    Grid:
        i, j = 0,...,N-1  (N x N grid)
        dx is the lattice spacing (default dx=1.0 per assignment hint)

    Initial conditions (assignment suggestion):
        u = 0.5 everywhere
        v = 0 outside a small center square
        v = 0.25 inside center square
        optional small noise

    Returns:
        dx, U0, V0, parameters dict
    """
    # Stability (explicit diffusion part): dt <= dx^2/(4Dmax) is a good rule of thumb
    Dmax = max(Du, Dv)
    if 4.0 * Dmax * dt / (dx**2) > 1.0:
        print(f"WARNING: diffusion CFL may be violated: 4*Dmax*dt/dx^2 = {4*Dmax*dt/(dx**2):.3f} > 1")

    U = 0.5 * np.ones((N, N), dtype=float)
    V = np.zeros((N, N), dtype=float)

    # center square perturbation for V
    r = max(2, N // 20)  # size of square relative to grid
    c = N // 2
    V[c - r:c + r, c - r:c + r] = 0.25

    # optional noise to break symmetry
    if noise > 0:
        U += noise * (np.random.rand(N, N) - 0.5)
        V += noise * (np.random.rand(N, N) - 0.5)

    # keep in [0,1] initially (not strictly required but helps avoid weird start)
    U = np.clip(U, 0.0, 1.0)
    V = np.clip(V, 0.0, 1.0)

    params = {"Du": Du, "Dv": Dv, "f": f, "k": k, "dt": dt, "dx": dx}
    return dx, U, V, params


# ============================================================
# 2) Laplacian with periodic boundaries (5-point stencil)
# ============================================================
def laplacian(Z, dx):
    """
    2D Laplacian with periodic BC in both directions using np.roll.
    """
    return (
        np.roll(Z, -1, axis=0) + np.roll(Z, 1, axis=0)
        + np.roll(Z, -1, axis=1) + np.roll(Z, 1, axis=1)
        - 4.0 * Z
    ) / (dx**2)


# ============================================================
# 3) One time step
# ============================================================
def gray_scott_step(U, V, Du, Dv, f, k, dt, dx):
    """
    Forward Euler update:
        U^{n+1} = U^n + dt*(Du Lap(U) - U V^2 + f(1-U))
        V^{n+1} = V^n + dt*(Dv Lap(V) + U V^2 - (f+k)V)
    """
    Lu = laplacian(U, dx)
    Lv = laplacian(V, dx)

    UV2 = U * V * V
    reaction_u = -UV2 + f * (1.0 - U)
    reaction_v =  UV2 - (f + k) * V

    U_new = U + dt * (Du * Lu + reaction_u)
    V_new = V + dt * (Dv * Lv + reaction_v)

    # optional: keep concentrations non-negative (helps with numerical blow-ups)
    U_new = np.clip(U_new, 0.0, 1.5)
    V_new = np.clip(V_new, 0.0, 1.5)

    return U_new, V_new


# ============================================================
# 4) Time evolution (store snapshots)
# ============================================================
def run_gray_scott(N, dt, n_steps, Du=0.16, Dv=0.08, f=0.035, k=0.060,
                   dx=1.0, noise=0.0, store_steps=None, seed=42):
    """
    Run Gray-Scott for n_steps.

    store_steps: list of integer time steps at which to store U,V.
    returns:
        U, V, snapshots(dict)
    """
    np.random.seed(seed)
    dx, U, V, params = initialize_gray_scott(N, dt, Du, Dv, f, k, dx=dx, noise=noise)

    if store_steps is None:
        store_steps = []
    store_set = set(store_steps)

    snapshots = {}
    snapshots[0] = (U.copy(), V.copy())

    for n in range(1, n_steps + 1):
        U, V = gray_scott_step(U, V, Du, Dv, f, k, dt, dx)
        if n in store_set:
            snapshots[n] = (U.copy(), V.copy())

    return U, V, snapshots, params


# ============================================================
# 5) Plotting helpers
# ============================================================
def plot_field(Z, title, output_file=None, show=True, vmin=None, vmax=None):

    fig, ax = plt.subplots(figsize=(8, 8), dpi=300)

    im = ax.imshow(Z, origin="lower", cmap="plasma", vmin=vmin, vmax=vmax)

    cbar = fig.colorbar(im, ax=ax, shrink=0.7, aspect=25, pad=0.02)
    cbar.ax.tick_params(labelsize=16)  # colorbar刻度更大

    # 只保留 f, k, step 信息
    ax.set_title(title, fontsize=26)

    ax.set_xlabel("x-index", fontsize=22)
    ax.set_ylabel("y-index", fontsize=22)

    # 坐标轴刻度字体变大
    ax.tick_params(axis='both', labelsize=20)

    plt.tight_layout()

    if output_file is not None:
        fig.savefig(output_file, dpi=400, bbox_inches="tight")

    if show:
        display(fig)

    plt.close(fig)

    
def animate_gray_scott(N, dt, n_steps, Du, Dv, f, k, dx=1.0, noise=0.0,
                       output_file="outputs/gray_scott.gif", fps=20, n_frames=150,
                       field="V", seed=42, show=True):
    """
    Make a GIF animation of U or V.
    field: "U" or "V"
    """
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    np.random.seed(seed)
    dx, U, V, params = initialize_gray_scott(N, dt, Du, Dv, f, k, dx=dx, noise=noise)

    frame_idx = np.linspace(0, n_steps, n_frames).astype(int)
    frame_set = set(frame_idx)

    frames = {}
    frames[0] = (U.copy(), V.copy())

    for n in range(1, n_steps + 1):
        U, V = gray_scott_step(U, V, Du, Dv, f, k, dt, dx)
        if n in frame_set:
            frames[n] = (U.copy(), V.copy())

    idx_sorted = sorted(frames.keys())
    Z0 = frames[idx_sorted[0]][0] if field == "U" else frames[idx_sorted[0]][1]

    fig, ax = plt.subplots(figsize=(6, 4), dpi=160)
    im = ax.imshow(Z0, origin="lower", cmap="plasma")
    fig.colorbar(im, ax=ax, label=f"{field} concentration")
    ax.set_xlabel("x-index")
    ax.set_ylabel("y-index")
    title = ax.set_title("")

    def update(k_):
        n = idx_sorted[k_]
        U_n, V_n = frames[n]
        Z = U_n if field == "U" else V_n
        im.set_data(Z)
        title.set_text(f"Gray-Scott ({field}), step={n}, t={n*dt:.2f}, f={f}, k={k}")
        return (im, title)

    ani = animation.FuncAnimation(fig, update, frames=len(idx_sorted), interval=60, blit=False)
    ani.save(output_file, writer=animation.PillowWriter(fps=fps))

    if show:
        display(HTML(ani.to_jshtml()))
    plt.close(fig)
    return ani


# ============================================================
# 6) Example run (recommended parameters from the assignment)
# ============================================================
if __name__ == "__main__":
    os.makedirs("outputs", exist_ok=True)

    # ---------------------------
    # Base simulation settings
    # ---------------------------
    N = 200
    dx = 1.0
    dt = 1.0

    Du = 0.16
    Dv = 0.08

    n_steps = 8000
    noise = 0.01
    seed = 42  # fixed for reproducibility

    # store snapshots for report figures
    store_steps = [0.00, 500, 1500, 3000, 8000]

    # ---------------------------
    # Three recommended (f,k) pairs
    # ---------------------------

    fk_list = [
        (0.035, 0.060),  # steady：kappa
        (0.035, 0.058),  # steady：theta
        (0.050, 0.060),  # steady：iota
        (0.025, 0.051),  # chaotic
        (0.025, 0.052),  # oscillating
        (0.030, 0.060),  # dissipation: V → 0
        #(0.035, 0.055),  # dissipation: V→initial 0.25
    ]

    # Use consistent color scaling across all plots (good for comparison in report)
    vmin, vmax = 0.0, 1.0

    for (f, k) in fk_list:
        tag = f"f{f:.3f}_k{k:.3f}".replace(".", "p")
        out_dir = os.path.join("outputs", tag)
        os.makedirs(out_dir, exist_ok=True)

        # Run simulation
        U, V, snaps, params = run_gray_scott(
            N=N, dt=dt, n_steps=n_steps,
            Du=Du, Dv=Dv, f=f, k=k,
            dx=dx, noise=noise,
            store_steps=store_steps,
            seed=seed
        )

        # Save snapshot figures (plot V, usually most informative)
        for n in store_steps:
            U_n, V_n = snaps[n]
            plot_field(
                V_n,
                title=f"Gray-Scott V | f={f}, k={k} | step={n}",
                output_file=os.path.join(out_dir, f"f={f}, k={k}, V_step{n}.png"),
                show=False,
                vmin=vmin,
                vmax=vmax
            )

        # Optional: make a GIF for each (f,k)
        animate_gray_scott(
            N=N, dt=dt, n_steps=n_steps,
            Du=Du, Dv=Dv, f=f, k=k,
            dx=dx, noise=noise,
            output_file=os.path.join(out_dir, f"f={f}, k={k}, V_animation.gif"),
            fps=20, n_frames=80, field="V",
            seed=seed,
            show=False
        )

        print(f"Done: f={f}, k={k} -> saved in {out_dir}")

Done: f=0.035, k=0.06 -> saved in outputs\f0p035_k0p060
Done: f=0.035, k=0.058 -> saved in outputs\f0p035_k0p058
Done: f=0.05, k=0.06 -> saved in outputs\f0p050_k0p060
Done: f=0.025, k=0.051 -> saved in outputs\f0p025_k0p051
Done: f=0.025, k=0.052 -> saved in outputs\f0p025_k0p052
Done: f=0.03, k=0.06 -> saved in outputs\f0p030_k0p060
